# SahelPay — Partie 2 : traitement batch avec SparkCore (RDD)

**Objectif** : manipuler l'API *bas niveau* de Spark (les RDD) sur un historique de 50 000 événements de transactions.

**Pré-requis** (déjà faits avant d'ouvrir ce notebook) :
- le fichier `transactions_history.jsonl` a été généré avec `producer.py --mode file --n 50000` ;
- il a été copié dans MinIO, dans `lake/bronze/history/`.

**Ce qu'on calcule**
1. Charger et parser le fichier.
2. Garder les transactions `COMPLETED`.
3. Montant total encaissé **par ville**.
4. Montant **moyen** par transaction et par ville (en un seul passage).
5. **Top 5** des commerçants par nombre de paiements validés.
6. Afficher le **lineage** et distinguer transformations / actions.

## 0. Les notions à avoir en tête

### Qu'est-ce qu'un RDD ?
**RDD = Resilient Distributed Dataset** : une collection de données **découpée en partitions** réparties sur les executors.
- *Distributed* : chaque partition est traitée en parallèle par un cœur.
- *Resilient* : si une partition est perdue (panne d'un worker), Spark sait la **recalculer** grâce au lineage (voir plus bas).
- Un RDD est **immuable** : on ne le modifie jamais, une transformation produit un *nouveau* RDD.

### Transformations et actions

| | Transformations | Actions |
|---|---|---|
| **Effet** | Décrivent un calcul, renvoient un **nouveau RDD** | Déclenchent le calcul, renvoient un **résultat** au Driver |
| **Exécution** | **Paresseuse** (*lazy*) : rien ne se passe | Exécution réelle sur le cluster |
| **Exemples** | `textFile`, `map`, `filter`, `reduceByKey`, `mapValues`, `aggregateByKey` | `count`, `collect`, `first`, `takeOrdered`, `take`, `sum` |

Spark attend donc la première *action* pour exécuter toute la chaîne d'un coup. Ça lui permet d'optimiser et d'enchaîner les étapes **en mémoire**, sans écrire sur disque entre chaque.

### Narrow vs wide : le shuffle
- **Transformation étroite** (`map`, `filter`) : chaque partition de sortie dépend d'**une seule** partition d'entrée → pas d'échange réseau.
- **Transformation large** (`reduceByKey`, `aggregateByKey`) : toutes les valeurs d'une même clé (ex. la ville `Rabat`) doivent se retrouver **dans la même partition** → les données sont **redistribuées** entre executors par le réseau. C'est le **shuffle**, l'opération la plus coûteuse de Spark.
- Chaque shuffle coupe le travail en deux **stages**.

### reduceByKey plutôt que groupByKey
`reduceByKey` **pré-agrège localement** dans chaque partition (*combiner*) avant le shuffle : on n'envoie sur le réseau qu'un total par ville et par partition, pas toutes les lignes. `groupByKey` enverrait tout. D'où le choix de `reduceByKey` ici.

### Lineage et DAG
- Le **lineage** est l'historique des transformations qui ont produit un RDD (`textFile → map → filter → map → reduceByKey`).
- Le **DAG** (graphe orienté acyclique) est ce même historique vu comme un graphe d'étapes, que Spark découpe en stages.
- C'est le lineage qui assure la **tolérance aux pannes** : si une partition disparaît, Spark rejoue uniquement les transformations nécessaires pour la reconstruire, sans avoir besoin de copies (contrairement à la réplication HDFS).

## 1. Démarrer la session

> ⚠️ Vérifie avant que l'ancienne session (Partie 1) est arrêtée : sinon elle garde les 4 cœurs et celle-ci restera en attente. Dans http://localhost:8080, la section *Running Applications* doit être vide.

In [1]:
import sys, json, time
sys.path.append("/workspace/scripts")          # pour retrouver spark_session.py
from spark_session import get_spark

spark = get_spark("SahelPay-Partie2")          # nom visible dans l'UI http://localhost:8080
sc = spark.sparkContext                         # SparkContext : l'objet qui crée les RDD

print("Spark", spark.version, "| master :", sc.master, "| appName :", sc.appName)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4e18d763-b61a-4b55-8bc7-9d6824960278;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr

Spark 3.5.3 | master : spark://spark-master:7077 | appName : SahelPay-Partie2


## 2. Charger le fichier et parser chaque ligne

`sc.textFile()` crée un RDD dont **chaque élément est une ligne** du fichier (du texte brut). Le fichier est lu **depuis MinIO** (`s3a://...`) car les executors tournent dans d'autres conteneurs : ils ne voient pas les fichiers de ton PC, mais tous voient MinIO.

On lui demande au moins 4 partitions (`minPartitions=4`) pour que les 4 cœurs du cluster travaillent. Sans ça, un fichier de 12 Mo tiendrait dans 1 ou 2 partitions et le cluster resterait sous-utilisé.

Puis `map(json.loads)` transforme chaque ligne (texte) en dictionnaire Python.

In [2]:
# Chemin du fichier dans MinIO : bucket "lake", dossier "bronze/history"
HISTORY_PATH = "s3a://lake/bronze/history/transactions_history.jsonl"

# TRANSFORMATION (paresseuse) : rien n'est lu pour le moment.
# minPartitions=4 : au moins 4 morceaux, donc 4 tâches en parallèle.
lines = sc.textFile(HISTORY_PATH, minPartitions=4)
print("Partitions du RDD 'lines' :", lines.getNumPartitions())

26/11/26 02:01:50 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Partitions du RDD 'lines' : 4


In [3]:
def parse(line):
    """Transforme une ligne JSON (texte) en dictionnaire Python."""
    return json.loads(line)

# TRANSFORMATION : on décrit le parsing, il sera exécuté plus tard, sur les executors.
events = lines.map(parse)

# cache() : garde le RDD en mémoire après son premier calcul, pour ne pas relire et
# re-parser le fichier à chaque question ci-dessous. (lui aussi paresseux)
events.cache()

# ACTION : count() déclenche réellement la lecture, le parsing et la mise en cache.
t0 = time.perf_counter()
nb_events = events.count()
temps = {"chargement": time.perf_counter() - t0}     # on note les temps pour le rapport (RDD vs DataFrame)

print(f"Événements lus : {nb_events:,}   (lecture + parsing : {temps['chargement']:.2f} s)")
print("Premier événement :")
print(events.first())                                  # ACTION : renvoie un élément au Driver

Événements lus : 50,000   (lecture + parsing : 34.30 s)
Premier événement :
{'tx_id': 'TX-00000001', 'type': 'P2M', 'payer_wallet': 'W-11494', 'payee_id': 'M-153', 'device_id': 'D-3ec0a2', 'city': 'Rabat', 'channel': 'QR', 'status': 'INITIATED', 'amount_mad': 12.37, 'event_time': '2026-11-25T11:28:32Z'}


## 3. Garder uniquement les transactions `COMPLETED`

Une transaction passe par plusieurs états (`INITIATED` → `AUTHORIZED` → `COMPLETED`, ou `REJECTED`, ou `REVERSED`) et **chaque changement d'état est un événement du fichier**. Pour compter l'argent réellement encaissé, on ne garde que les `COMPLETED`.

`filter` est une transformation **étroite** : chaque partition est filtrée indépendamment, sans échange réseau.

In [4]:
# TRANSFORMATION étroite : on garde les événements dont le statut est COMPLETED.
completed = events.filter(lambda e: e["status"] == "COMPLETED")
completed.cache()    # on va s'en servir pour toutes les questions suivantes

# Pour comprendre les données : combien d'événements par statut ?
# map -> paires (statut, 1), reduceByKey -> on additionne les 1 de chaque statut.
par_statut = events.map(lambda e: (e["status"], 1)).reduceByKey(lambda a, b: a + b)

print("Événements par statut :")
for statut, n in sorted(par_statut.collect()):         # collect() = ACTION : ramène le résultat au Driver
    print(f"  {statut:<11} {n:>7,}")
print(f"\nTransactions COMPLETED : {completed.count():,}")

Événements par statut :


  AUTHORIZED   16,014
  COMPLETED    15,114
  INITIATED    16,840
  REJECTED      1,694
  REVERSED        338

Transactions COMPLETED : 15,114


## 4. Montant total encaissé par ville

Le schéma classique `map` + `reduceByKey` :
1. `map` : chaque événement devient une paire **(clé, valeur)** = `(ville, montant)`.
2. `reduceByKey` : pour chaque ville, on additionne les montants.

`reduceByKey` est une transformation **large** : c'est elle qui provoque le **shuffle** et donc une coupure en 2 stages. Pour chaque ville, Spark additionne d'abord les montants *dans chaque partition* (pré-agrégation locale), puis envoie ces sous-totaux sur le réseau et les additionne.

In [5]:
t0 = time.perf_counter()

# 1) map : événement -> (ville, montant)        [transformation étroite]
villes_montants = completed.map(lambda e: (e["city"], e["amount_mad"]))

# 2) reduceByKey : on additionne les montants de chaque ville   [transformation LARGE = shuffle]
total_par_ville = villes_montants.reduceByKey(lambda a, b: a + b)

# 3) collect : ACTION, qui déclenche tout le calcul. Il y a peu de villes, le résultat est petit.
resultat_total = total_par_ville.collect()
temps["total_par_ville"] = time.perf_counter() - t0

print(f"{'Ville':<12} {'Montant total (MAD)':>20}")
print("-" * 33)
for ville, total in sorted(resultat_total, key=lambda x: -x[1]):    # tri décroissant sur le montant
    print(f"{ville:<12} {total:>20,.2f}")
print(f"\nTemps de calcul : {temps['total_par_ville']:.2f} s")

Ville         Montant total (MAD)
---------------------------------
Casablanca             899,636.66
Rabat                  391,910.51
Marrakech              256,105.94
Fes                    203,855.98
Tanger                 195,252.06
Kenitra                158,493.69
Meknes                 139,409.05
Agadir                 122,899.06
Oujda                  106,596.76
Tetouan                105,688.33

Temps de calcul : 0.97 s


## 5. Montant moyen par transaction et par ville, en un seul passage

Une moyenne = **somme ÷ nombre**. Piège classique : on **ne peut pas** faire la moyenne des moyennes de chaque partition (les partitions n'ont pas la même taille). Il faut additionner séparément les montants **et** les nombres de transactions, puis diviser à la fin.

Astuce « en un seul passage » : on fait voyager un couple **(somme, nombre)** à la place d'un simple montant, de sorte qu'un seul `reduceByKey` calcule les deux d'un coup.
- Chaque événement devient `(ville, (montant, 1))`.
- Le `reduceByKey` additionne terme à terme : `(s1, n1) + (s2, n2) = (s1+s2, n1+n2)`.
- `mapValues` calcule enfin `somme / nombre`.

On montre ensuite la version `aggregateByKey`, plus explicite, qui distingue la fusion *dans* une partition de la fusion *entre* partitions.

In [6]:
t0 = time.perf_counter()

# Version 1 : reduceByKey avec un couple (somme, nombre)
somme_nombre = (completed
    .map(lambda e: (e["city"], (e["amount_mad"], 1)))              # (ville, (montant, 1))
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))         # addition terme à terme

# mapValues : ne touche qu'aux valeurs (pas aux clés) -> transformation étroite, pas de shuffle
moyenne_par_ville = somme_nombre.mapValues(lambda sn: sn[0] / sn[1])

# ACTION : on ramène (ville, (somme, nombre)) pour afficher tout dans un seul tableau
resultat_sn = somme_nombre.collect()
temps["moyenne_par_ville"] = time.perf_counter() - t0

print(f"{'Ville':<12} {'Total (MAD)':>16} {'Nb trans.':>10} {'Moyenne (MAD)':>15}")
print("-" * 56)
for ville, (somme, nombre) in sorted(resultat_sn, key=lambda x: -x[1][0]):
    print(f"{ville:<12} {somme:>16,.2f} {nombre:>10,} {somme / nombre:>15,.2f}")
print(f"\nTemps de calcul : {temps['moyenne_par_ville']:.2f} s")

Ville             Total (MAD)  Nb trans.   Moyenne (MAD)
--------------------------------------------------------
Casablanca         899,636.66      4,715          190.80
Rabat              391,910.51      2,515          155.83
Marrakech          256,105.94      1,436          178.35
Fes                203,855.98      1,318          154.67
Tanger             195,252.06      1,247          156.58
Kenitra            158,493.69        950          166.84
Meknes             139,409.05        876          159.14
Agadir             122,899.06        696          176.58
Oujda              106,596.76        683          156.07
Tetouan            105,688.33        678          155.88

Temps de calcul : 0.58 s


In [7]:
# Version 2 : aggregateByKey (même résultat, avec trois éléments bien séparés)
agregat = (completed
    .map(lambda e: (e["city"], e["amount_mad"]))
    .aggregateByKey(
        (0.0, 0),                                   # valeur initiale : (somme=0, nombre=0)
        lambda acc, montant: (acc[0] + montant, acc[1] + 1),   # seqOp : ajoute UN montant DANS une partition
        lambda a, b: (a[0] + b[0], a[1] + b[1])))              # combOp : fusionne les résultats de 2 partitions

# Vérification : les deux méthodes doivent donner les mêmes sommes et nombres.
v1 = dict(somme_nombre.collect())
v2 = dict(agregat.collect())
ok = all(abs(v1[v][0] - v2[v][0]) < 1e-6 and v1[v][1] == v2[v][1] for v in v1)
print("reduceByKey et aggregateByKey donnent le même résultat :", ok)

reduceByKey et aggregateByKey donnent le même résultat : True


## 6. Top 5 des commerçants par nombre de paiements validés

- On ne garde que les paiements chez un commerçant : `type == "P2M"` (pour un `P2P`, le bénéficiaire est un particulier, pas un commerçant).
- `map` → `(payee_id, 1)`, `reduceByKey` → nombre de paiements par commerçant.
- `takeOrdered(5, key=...)` renvoie les 5 premiers selon la clé de tri **sans trier tout le RDD** : chaque partition garde ses 5 meilleurs, et le Driver en choisit 5 parmi ceux-là.

Le tri se fait sur `-nombre` (décroissant), puis sur l'identifiant pour départager les ex æquo de façon stable.

In [8]:
t0 = time.perf_counter()

# Paiements validés chez un commerçant : (id_commerçant, nombre de paiements)
paiements_par_commercant = (completed
    .filter(lambda e: e["type"] == "P2M")              # paiements commerçants uniquement
    .map(lambda e: (e["payee_id"], 1))                 # (M-xxx, 1)
    .reduceByKey(lambda a, b: a + b))                  # total par commerçant  [shuffle]

# ACTION : takeOrdered. key = (-nombre, id) => plus grand nombre d'abord, id pour départager.
top5 = paiements_par_commercant.takeOrdered(5, key=lambda kv: (-kv[1], kv[0]))
temps["top5_commercants"] = time.perf_counter() - t0

print("Top 5 des commerçants (paiements validés) :")
for rang, (commercant, n) in enumerate(top5, start=1):
    print(f"  {rang}. {commercant}   {n} paiements")
print(f"\nTemps de calcul : {temps['top5_commercants']:.2f} s")

Top 5 des commerçants (paiements validés) :
  1. M-032   48 paiements
  2. M-215   48 paiements
  3. M-126   47 paiements
  4. M-254   47 paiements
  5. M-136   46 paiements

Temps de calcul : 0.50 s


## 7. Lineage : `toDebugString()`

`toDebugString()` affiche l'**historique des transformations** d'un RDD, de la source (en bas) jusqu'au RDD final (en haut). Il ne lance aucun calcul.

**Comment le lire**
- Chaque ligne est un RDD, le plus récent en haut.
- Le nombre entre parenthèses `(4)` = nombre de **partitions**.
- Un changement d'indentation avec `+-` marque une **frontière de shuffle**, c'est-à-dire une coupure entre deux **stages**.
- `ShuffledRDD` = le résultat d'un `reduceByKey`.
- `PythonRDD` = les `map` / `filter` exécutés côté Python.
- Les mentions `CachedPartitions` indiquent les RDD mis en cache.

In [9]:
# toDebugString() renvoie des octets (bytes) en PySpark : on les décode en texte.
print("=== Lineage : montant total par ville ===")
print(total_par_ville.toDebugString().decode("utf-8"))

print("\n=== Lineage : top 5 des commerçants (avant l'action takeOrdered) ===")
print(paiements_par_commercant.toDebugString().decode("utf-8"))

=== Lineage : montant total par ville ===
(4) PythonRDD[16] at collect at /tmp/ipykernel_1897/3090534013.py:10 []
 |  MapPartitionsRDD[15] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[14] at partitionBy at <unknown>:0 []
 +-(4) PairwiseRDD[13] at reduceByKey at /tmp/ipykernel_1897/3090534013.py:7 []
    |  PythonRDD[12] at reduceByKey at /tmp/ipykernel_1897/3090534013.py:7 []
    |  PythonRDD[5] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 4; MemorySize: 725.0 KiB; DiskSize: 0.0 B
    |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 4; MemorySize: 1444.9 KiB; DiskSize: 0.0 B
    |  s3a://lake/bronze/history/transactions_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
    |  s3a://lake/bronze/history/transactions_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []

=== Lineage : top 5 des commerçants (avant l'action takeOrdered) ===
(4) PythonRDD[32] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[30] at

### Transformations (lazy) et actions dans ce notebook

| Instruction | Type | Remarque |
|---|---|---|
| `sc.textFile(...)` | Transformation | ne lit rien |
| `.map(...)` / `.mapValues(...)` | Transformation étroite | pas de shuffle |
| `.filter(...)` | Transformation étroite | pas de shuffle |
| `.reduceByKey(...)` / `.aggregateByKey(...)` | Transformation **large** | **shuffle**, coupure en 2 stages |
| `.cache()` | Marqueur (lazy) | la mise en mémoire a lieu à la 1re action |
| `.count()` / `.first()` / `.collect()` / `.takeOrdered()` | **Action** | déclenchent le calcul |
| `.toDebugString()` | ni l'un ni l'autre | affiche seulement le plan |

## 8. Bonus : l'effet des doublons

Le générateur duplique environ **2 %** des événements (même `tx_id` et même `status`), comme le ferait un réseau qui renvoie un message. Dans les calculs ci-dessus, **ces doublons sont comptés deux fois**. L'énoncé ne demande pas de les retirer en Partie 2, mais il est utile de mesurer ce que ça change : en Partie 5, on les supprimera avec `dropDuplicates(["tx_id", "status"])`.

In [10]:
# Dédoublonnage en RDD : on prend (tx_id, status) comme clé et on garde un seul événement par clé.
completed_uniques = (completed
    .map(lambda e: ((e["tx_id"], e["status"]), e))
    .reduceByKey(lambda a, b: a)               # entre deux doublons, on en garde un
    .values())                                 # on retire la clé, il reste l'événement

n_avec = completed.count()
n_sans = completed_uniques.count()
print(f"COMPLETED avec doublons : {n_avec:,}")
print(f"COMPLETED sans doublons : {n_sans:,}")
print(f"Doublons détectés       : {n_avec - n_sans:,}  ({(n_avec - n_sans) / n_avec:.1%})")

montant_avec = completed.map(lambda e: e["amount_mad"]).sum()
montant_sans = completed_uniques.map(lambda e: e["amount_mad"]).sum()
print(f"\nMontant total avec doublons : {montant_avec:,.2f} MAD")
print(f"Montant total sans doublons : {montant_sans:,.2f} MAD")
print(f"Surestimation               : {montant_avec - montant_sans:,.2f} MAD")

COMPLETED avec doublons : 15,114
COMPLETED sans doublons : 14,825
Doublons détectés       : 289  (1.9%)

Montant total avec doublons : 2,579,848.04 MAD
Montant total sans doublons : 2,532,167.97 MAD
Surestimation               : 47,680.07 MAD


## 9. Récapitulatif des temps (à conserver pour la Partie 3)

Le rapport de la Partie 3 demande de **comparer RDD et DataFrame avec les temps mesurés**. On garde donc les temps de cette partie. Ils dépendent de la machine et du cache : l'important est de refaire les mêmes calculs en Partie 3 dans les mêmes conditions.

In [11]:
print(f"{'Étape':<28} {'Temps (s)':>10}")
print("-" * 40)
for etape, secondes in temps.items():
    print(f"{etape:<28} {secondes:>10.2f}")

Étape                         Temps (s)
----------------------------------------
chargement                        34.30
total_par_ville                    0.97
moyenne_par_ville                  0.58
top5_commercants                   0.50


In [14]:
# Mesure "à chaud" : on répète 3 fois et on garde la médiane (plus stable qu'une seule mesure).
def chronometrer(calcul, n=3):
    durees = []
    for _ in range(n):
        t0 = time.perf_counter()
        calcul()                                    # exécute le calcul complet (avec son action)
        durees.append(time.perf_counter() - t0)
    return sorted(durees)[n // 2]                   # médiane

temps["total_par_ville (à chaud)"] = chronometrer(
    lambda: completed.map(lambda e: (e["city"], e["amount_mad"]))
                     .reduceByKey(lambda a, b: a + b).collect())

temps["moyenne_par_ville (à chaud)"] = chronometrer(
    lambda: completed.map(lambda e: (e["city"], (e["amount_mad"], 1)))
                     .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])).collect())

## 10. Captures et réponses pour le rapport

### Captures à faire **avant d'arrêter la session**
L'interface du Driver (port 4040) n'existe que tant que la session est active.
1. Ouvre **http://localhost:4040** → onglet **Jobs**.
2. Clique sur le job du calcul *montant total par ville* → **DAG Visualization** : capture le DAG et note le **nombre de stages**.
3. Clique sur un *Stage* pour voir ses tâches, et repère le **Shuffle Write** (stage 1) et le **Shuffle Read** (stage 2).
4. Capture aussi **http://localhost:8080** avec `SahelPay-Partie2` dans *Running Applications*.

### À expliquer dans le rapport (avec tes mots)
- **Nombre de stages** de ton job et pourquoi : un shuffle coupe le travail en 2 stages (lecture/filtrage/map, puis agrégation finale).
- **Le shuffle de `reduceByKey`** : toutes les valeurs d'une même ville doivent atterrir dans la même partition, donc les données circulent entre executors. La pré-agrégation locale réduit ce qui est envoyé.
- **Lineage et tolérance aux pannes** : Spark mémorise la chaîne de transformations ; si un worker tombe, il recalcule seulement les partitions perdues en rejouant cette chaîne depuis la source.
- **Transformations vs actions** (tableau de la section 7).

## 11. Libérer le cluster

> Exécute cette cellule **seulement après** avoir fait tes captures sur http://localhost:4040.

In [15]:
# Arrête la session et libère les 4 cœurs du cluster.
spark.stop()